<a href="https://colab.research.google.com/github/legna7816/ml-projects/blob/main/nlp_sentiment/nlp_step06_BERT_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install transformers torch datasets scikit-learn # 최초 1회 설치

import torch
import pandas as pd
import numpy as np
import os

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# GPU 확인 (cuda가 나와야 정상)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 디바이스:', device)

사용 디바이스: cuda


In [2]:
# 1. 데이터 불러오기 (이전에 썼던 IMDB, 없으면 재다운로드)
if not os.path.exists('aclImdb'):
    !wget -q http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
    !tar -xzf aclImdb_v1.tar.gz

def load_imdb(path, split='train'):
  texts, labels = [], []
  for label, sentiment in enumerate(['neg', 'pos']):
    folder = os.path.join(path, split, sentiment)
    for fname in os.listdir(folder):
      with open(os.path.join(folder, fname), encoding='utf-8') as f:
        texts.append(f.read())
      labels.append(label)
  return texts, labels

train_texts, train_labels = load_imdb('aclImdb', 'train')

# 파인튜닝은 학습이 오래 걸리므로 일부만 샘플링해서 실습
df = pd.DataFrame({'text': train_texts, 'label': train_labels})
df_sample = df.sample(2000, random_state=42)

train_df, val_df = train_test_split(df_sample, test_size=0.2, random_state=42)
print(train_df.shape, val_df.shape)

(1600, 2) (400, 2)


In [3]:
# 2. 토큰화 - 텍스트를 BERT 입력 형태로 변환
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_texts(texts, max_length=256):
  # truncation=True: max_length 넘으면 자름 (리뷰가 김, 전체 다 넣으면 느려짐)
  # padding=True: 문장 길이가 다 달라서, 짧은 문장은 빈 공간을 채워 길이를 맞춤
  return tokenizer(texts, truncation=True, padding=True, max_length=max_length)

train_encodings = tokenize_texts(train_df['text'].tolist())
val_encodings = tokenize_texts(val_df['text'].tolist())

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [4]:
# 3. PyTorch Dataset 형태로 감싸기 (Trainer가 요구하는 형식)
class IMDBDataset(torch.utils.data.Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels

  def __getitem__(self, idx):
    item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx])
    return item

  def __len__(self):
    return len(self.labels)

train_dataset = IMDBDataset(train_encodings, train_df['label'].tolist())
val_dataset = IMDBDataset(val_encodings, val_df['label'].tolist())

In [5]:
# 4. 모델 불러오기 - 분류용 헤드가 얹힌 BERT
# AutoModel과 다르게 ForSequenceClassification은
# BERT 위에 "분류를 위한 작은 레이어"가 자동으로 추가된 버전
# num_labels=2; 긍정/부정 -> 2개 클래스
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)    # 모델을 GPU로 이동

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [6]:
# 학습 설정 & 파인튜닝 실행
def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis=1)
  return {'accuracy': accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,             # 전체 데이터를 2번 반복 학습
    per_device_train_batch_size=8,   # 한 번에 8개씩 묶어서 학습
    per_device_eval_batch_size=8,
    eval_strategy='epoch',           # 매 epoch마다 검증
    save_strategy='no',              # 실습이라 중간 저장 생략 (속도↑)
    logging_steps=20,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()    # 실제 학습 시작

Epoch,Training Loss,Validation Loss,Accuracy
1,0.433102,0.478393,0.805000
2,0.238712,0.521114,0.860000


TrainOutput(global_step=400, training_loss=0.36355905890464785, metrics={'train_runtime': 156.6177, 'train_samples_per_second': 20.432, 'train_steps_per_second': 2.554, 'total_flos': 420977688576000.0, 'train_loss': 0.36355905890464785, 'epoch': 2.0})

In [7]:
# 6. 평가 & 비교
eval_result = trainer.evaluate()
print('파인튜닝된 BERT 정확도:', eval_result['eval_accuracy'])
# 비교: TF-IDF + LogisticRegression은 10000개 데이터로 86.4%였음
# 여기는 2000개로도 비슷하거나 더 나은 성능이 나오는지 확인

Training Loss,Validation Loss,Epoch,Accuracy
0.238712,0.521114,2,0.860000


파인튜닝된 BERT 정확도: 0.86


In [8]:
# 7. 새 리뷰로 직접 예측
def predict_sentiment(text):
  inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True).to(device)
  with torch.no_grad():
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    return '긍정' if pred == 1 else '부정'

print(predict_sentiment("this movie was absolutely fantastic and the acting was superb"))
print(predict_sentiment("terrible film boring and waste of time"))
print(predict_sentiment("the movie was not good at all"))    # not이 들어간 부정 표현 테스트

긍정
부정
부정


In [9]:
# 8. TODO
# 8-1. BERT로 "the movie was not bad" 확인
print(predict_sentiment("the movie was not bad"))

긍정


In [10]:
# 8-2. num_train_epochs를 2 -> 1로 줄였을 때 정확도 비교
model_1 = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model_1.to(device)

training_args_1 = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,    # 1로 변경
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=20,
)

trainer_1 = Trainer(
    model=model_1,
    args=training_args_1,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer_1.train()
eval_result_1 = trainer_1.evaluate()
print('num_train_epochs를 1로 변경한  BERT 정확도:', eval_result_1['eval_accuracy'])
print('파인튜닝된 기존 BERT 정확도:', eval_result['eval_accuracy'])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.301571,0.402328,0.867500


Training Loss,Validation Loss,Epoch,Accuracy
0.301571,0.402328,1,0.867500


num_train_epochs를 1로 변경한  BERT 정확도: 0.8675
파인튜닝된 기존 BERT 정확도: 0.86


In [11]:
# 8-3. df_sample 크기를 2000 -> 500으로 줄이면 정확도와 학습 시간이 어떻게 변하는지 확인
df_sample_2 = df.sample(500, random_state=42)

train_df_2, val_df_2 = train_test_split(df_sample_2, test_size=0.2, random_state=42)

train_encodings_2 = tokenize_texts(train_df_2['text'].tolist())
val_encodings_2 = tokenize_texts(val_df_2['text'].tolist())

train_dataset_2 = IMDBDataset(train_encodings_2, train_df_2['label'].tolist())
val_dataset_2 = IMDBDataset(val_encodings_2, val_df_2['label'].tolist())

model_2 = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model_2.to(device)

training_args_2 = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=20,
)

trainer_2 = Trainer(
    model=model_2,
    args=training_args_2,
    train_dataset=train_dataset_2,
    eval_dataset=val_dataset_2,
    compute_metrics=compute_metrics,
)

trainer_2.train()

eval_result_2 = trainer_2.evaluate()
print('df_sample size가 500인 BERT 정확도:', eval_result_2['eval_accuracy'])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.601476,0.462872,0.760000
2,0.295367,0.303903,0.900000


Training Loss,Validation Loss,Epoch,Accuracy
0.295367,0.303903,2,0.900000


df_sample size가 500인 BERT 정확도: 0.9
